In [17]:
import pandas as pd

In [18]:
def read_primary_name():
    df = pd.read_csv("../data/input/primary_name.csv")
    return df 


def read_ori():
    df = pd.read_csv("../data/input/ori.csv")
    return df

def read_npi():
    df = pd.read_csv("../data/input/california_npi_output.csv")
    return df

primary_name_df = read_primary_name()

ori_df = read_ori()

ori_df


merged_df = pd.merge(primary_name_df, ori_df, on="primary_name")

merged_df = merged_df[["post_name", "ORI"]]

merged_df = merged_df.rename(columns={"post_name": "agency_name", "ORI": "agency_ori"})

## we drop from 800~ rows to 627 rows when we merge, that said 627 sounds more correct than 800 agencies with ORI
merged_df.shape

(627, 2)

In [19]:

ca_npi = read_npi()

### not all agencies have an ORI. porque? rows are dropped when we merge npi with katies ori data
df = pd.merge(ca_npi, merged_df, on="agency_name")

## then we drop to 595 agencies when we match ca_npi with katey's ori data
df.agency_name.nunique()

missing_from_npi = merged_df[~merged_df['agency_name'].isin(ca_npi['agency_name'])][['agency_name', 'agency_ori']].drop_duplicates()
missing_from_npi = missing_from_npi.rename(columns={'agency_name': 'katey_agency_name', 'agency_ori': 'katey_agency_ori'})
missing_from_npi['npi_agency_name'] = 'No match'
missing_from_npi['source'] = 'Missing from NPI'

missing_from_katey = ca_npi[~ca_npi['agency_name'].isin(merged_df['agency_name'])]['agency_name'].drop_duplicates().reset_index(drop=True)
missing_from_katey_df = pd.DataFrame({
    'katey_agency_name': 'No match',
    'katey_agency_ori': 'No match', 
    'npi_agency_name': missing_from_katey,
    'source': 'Missing from Katey'
})

# Combine both dataframes
all_missing = pd.concat([missing_from_npi, missing_from_katey_df], ignore_index=True)

print(f"Total mismatched agencies: {all_missing.shape[0]}")
print(f"Missing from NPI: {missing_from_npi.shape[0]}")
print(f"Missing from Katey: {missing_from_katey_df.shape[0]}")
# all_missing.to_csv("../data/output/mismatch_npi_katey.csv", index=False)

Total mismatched agencies: 272
Missing from NPI: 18
Missing from Katey: 254


/var/folders/r9/3_1rmy995xs_9z4vz66rsf9r0000gn/T/ipykernel_42931/2238663651.py:11: DtypeWarning: Columns (0,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/input/california_npi_output.csv")


In [ ]:
def read_ori_national():
    df = pd.read_csv("../data/input/da35158-0001.csv")
    df = df[df.ADDRESS_STATE.str.contains("CA")]	

    # df = df[["ORI9", "NAME","COUNTYNAME", "UANAME","AGCYTYPE", "LG_NAME", "ADDRESS_NAME", "ADDRESS_STR1", "ADDRESS_CITY",  "ADDRESS_ZIP", "LG_POPULATION", "INTPTLAT", "INTPTLONG"]] 	

    df = df.rename(columns={"ORI9": "agency_ori"})		

    df = df[df.agency_ori.astype(str).str.contains(r"CA0388900")]
					
    return df 

df_ori_national = read_ori_national()
# ### missing 100k rows, because of missing ORI? 
# df_merged = pd.merge(df, df_ori_national, on="agency_ori")

# # we drop to 439 rows when we merge national ORI with katey's ORI. Why? which agencies in katey's data don't match with national?
# df_merged.agency_name.nunique()

# # Agencies in df but not in national data
# missing_from_national = df[~df['agency_ori'].isin(df_ori_national['agency_ori'])][['agency_name', 'agency_ori']].drop_duplicates()
# missing_from_national = missing_from_national.rename(columns={'agency_name': 'katey_agency_name', 'agency_ori': 'katey_agency_ori'})
# missing_from_national['national_agency_name'] = 'No match'
# missing_from_national['national_agency_ori'] = 'No match'
# missing_from_national['source'] = 'Missing from National'

# # Agencies in national data but not in df
# missing_from_katey = df_ori_national[~df_ori_national['agency_ori'].isin(df['agency_ori'])][['NAME', 'agency_ori']].drop_duplicates()
# missing_from_katey_df = pd.DataFrame({
#     'katey_agency_name': 'No match',
#     'katey_agency_ori': 'No match',
#     'national_agency_name': missing_from_katey['NAME'],
#     'national_agency_ori': missing_from_katey['agency_ori'],
#     'source': 'Missing from Katey'
# }).reset_index(drop=True)

# # Combine both dataframes
# all_missing_national = pd.concat([missing_from_national, missing_from_katey_df], ignore_index=True)

# print(f"Total mismatched agencies: {all_missing_national.shape[0]}")
# print(f"Missing from National: {missing_from_national.shape[0]}")
# print(f"Missing from Katey: {missing_from_katey_df.shape[0]}")
# all_missing_national.to_csv("../data/output/missing_npi_katey_w_national.csv", index=False)